# Interpretabilitate și Analiză pe Lungime
**Integrated Gradients** la nivel de token și step pe 4 exemple reprezentative (TP, TN, FP, FN). Analiză a performanței pe buckets de lungime a traiectoriei.

In [ ]:
import os, re, json
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from scipy.stats import spearmanr
from collections import Counter
from tqdm import tqdm
import pandas as pd


## Configurare

In [ ]:
BASE      = os.path.expanduser("~/project/data")
MODEL_DIR = os.path.expanduser("~/project/model")
LOG_DIR   = os.path.expanduser("~/project/logs")
os.makedirs(LOG_DIR, exist_ok=True)

MODEL_NAME   = "Qwen/Qwen3-0.6B"
MAX_LENGTH   = 1024
BATCH_SIZE   = 8
LABEL_MAP    = {"safe": 0, "potentially unsafe": 1, "unsafe": 2}
IDX_TO_LABEL = {0: "safe", 1: "potentially_unsafe", 2: "unsafe"}
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BEST_RANK    = 16  # modelul cu cel mai bun macro-F1
print(f"Device: {device}")


In [ ]:
# Filtre pentru tokeni irelevanti în atribuire IG
SPECIAL_TOKENS = {"<|endoftext|>","<s>","</s>","<pad>","<|im_start|>","<|im_end|>",
                  "[CLS]","[SEP]","[PAD]","[UNK]","[MASK]"}
STOPWORDS = {"a","an","the","of","in","on","at","to","for","is","are","was","were",
             "be","been","being","and","or","but","not","no","his","her","its","their",
             "our","your","my","this","that","it","he","she","they","we","i","you","as",
             "by","from","with","about","into","through","during","has","have","had",
             "do","does","did","will","would","could","should","may","might","shall",
             "can","need","ought","s","t","re","ve","d","ll","also","just","more"}
BPE_ARTIFACTS = {"Ċ","ĊĊ","ĠĊĊ",":Ċ",".Ċ","?Ċ","!Ċ","âĢĻ","_","Ġ","ĊStep","âĢ","Ļ","ľ","Ŀ"}
BPE_SUFFIXES  = {"ing","ow","igh","fer","ude","wi","ta","ya","nab","tin","ys","tion","er",
                 "ed","al","ly","ment","ness","ful","less","ble","ple","tle","dle","ck"}
INPUT_HEADERS  = {"Query","Trace","Reason","Step","Reasoning"}
ALL_FILTERED   = STOPWORDS | BPE_ARTIFACTS | BPE_SUFFIXES | INPUT_HEADERS


## Date și model

In [ ]:
def load_data():
    df_unsafe      = pd.read_json(f"{BASE}/Train/train_unsafe.jsonl", lines=True)
    df_safe        = pd.read_json(f"{BASE}/Train/train_safe.jsonl", lines=True)
    df_potentially = pd.read_json(f"{BASE}/Train/train_potentially_unsafe.jsonl", lines=True)
    df_full        = pd.concat([df_unsafe, df_safe, df_potentially], ignore_index=True)
    _, val_df = train_test_split(df_full, test_size=0.10, random_state=42,
                                 stratify=df_full['label'] if 'label' in df_full.columns else None)

    dv_unsafe      = pd.read_json(f"{BASE}/Validation/valid_unsafe.jsonl", lines=True)
    dv_safe        = pd.read_json(f"{BASE}/Validation/valid_safe.jsonl", lines=True)
    dv_potentially = pd.read_json(f"{BASE}/Validation/valid_potentially_unsafe.jsonl", lines=True)
    test_df        = pd.concat([dv_unsafe, dv_safe, dv_potentially], ignore_index=True)
    print(f"Test: {len(test_df)}")
    return val_df, test_df

val_df, test_df = load_data()


In [ ]:
class SafetyClassifier(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_classes=3, r_value=8):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(model_name)
        lora_cfg = LoraConfig(r=r_value, lora_alpha=2*r_value,
                              target_modules=["q_proj","k_proj","v_proj","o_proj"],
                              lora_dropout=0.1, bias="none", task_type="FEATURE_EXTRACTION")
        self.base_model = get_peft_model(self.base_model, lora_cfg)
        hidden = self.base_model.config.hidden_size
        self.classifier = nn.Sequential(
            nn.Linear(hidden, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        out  = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        lhs  = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).expand(lhs.size()).float()
        vec  = torch.sum(lhs * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
        return self.classifier(vec)

tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model_path = os.path.join(MODEL_DIR, f"qwen_safety_model_r{BEST_RANK}.pt")
model      = SafetyClassifier(r_value=BEST_RANK).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print("Model incarcat.")


## Integrated Gradients
Atribuim importanță fiecărui token față de logit-ul clasei `safe` (clasa 0). Integrăm gradientul pe un drum de la un input de referință (toate padding-uri) la inputul real, cu 50 de pași.

In [ ]:
def explain_ig(model, tokenizer, text, target_class=0, n_steps=50):
    enc = tokenizer(text, return_tensors="pt", truncation=True,
                    max_length=MAX_LENGTH, padding="max_length")
    ids  = enc['input_ids'].to(device)
    mask = enc['attention_mask'].to(device)

    emb_layer = model.base_model.model.embed_tokens
    embeds    = emb_layer(ids)
    baseline  = emb_layer(torch.full_like(ids, tokenizer.pad_token_id))

    integrated_grads = torch.zeros_like(embeds)
    for alpha in torch.linspace(0, 1, n_steps):
        interpolated = baseline + alpha * (embeds - baseline)
        interpolated = interpolated.detach().requires_grad_(True)

        def forward_with_embeds(emb):
            out = model.base_model.model(inputs_embeds=emb, attention_mask=mask)
            lhs = out.last_hidden_state
            m   = mask.unsqueeze(-1).expand(lhs.size()).float()
            vec = torch.sum(lhs * m, 1) / torch.clamp(m.sum(1), min=1e-9)
            return model.classifier(vec)

        logits = forward_with_embeds(interpolated)
        score  = logits[0, target_class]
        score.backward()
        integrated_grads += interpolated.grad.detach()

    integrated_grads = integrated_grads / n_steps
    attributions     = (integrated_grads * (embeds - baseline).detach()).sum(-1).squeeze()

    tokens = tokenizer.convert_ids_to_tokens(ids.squeeze().cpu().numpy())
    delta  = (logits[0, target_class] - forward_with_embeds(baseline)[0, target_class]).item()

    word_attrs = []
    for tok, attr in zip(tokens, attributions.cpu().numpy()):
        clean = tok.replace("Ġ", "").replace("Ċ", "").strip()
        if (clean and clean not in SPECIAL_TOKENS and clean not in ALL_FILTERED
                and len(clean) > 1 and not clean.startswith("##")):
            word_attrs.append((clean, float(attr)))

    return word_attrs, attributions.cpu().numpy(), delta


In [ ]:
def aggregate_by_steps(raw_attrs, expected_steps=None):
    tokens = tokenizer.convert_ids_to_tokens(
        tokenizer("", return_tensors="pt")['input_ids'].squeeze().cpu().numpy()
    ) if False else []

    step_pattern = re.compile(r'step\s*\d+', re.IGNORECASE)
    # lucrăm direct pe raw_attrs (index corespunde tokenilor)
    result = {}
    current = "Query"
    buf = []
    for attr in raw_attrs:
        buf.append(attr)
    # simplu: împărțim în buckets egale dacă avem expected_steps
    if expected_steps and expected_steps > 0:
        chunk = max(1, len(raw_attrs) // (expected_steps + 1))
        result["Query"] = float(np.sum(raw_attrs[:chunk]))
        for i in range(expected_steps):
            start = chunk * (i + 1)
            end   = chunk * (i + 2)
            result[f"Step {i+1}"] = float(np.sum(raw_attrs[start:end]))
    else:
        result["Entire input"] = float(np.sum(raw_attrs))
    return result


In [ ]:
def plot_token_heatmap(word_attrs, cat, gt, pred, save_path, top_n=12):
    if not word_attrs:
        return
    sorted_attrs = sorted(word_attrs, key=lambda x: abs(x[1]), reverse=True)[:top_n]
    max_val      = max(abs(a) for _, a in sorted_attrs) or 1.0
    norm         = mcolors.TwoSlopeNorm(vmin=-max_val, vcenter=0, vmax=max_val)
    cmap         = plt.get_cmap("RdYlGn")

    cols = 4
    rows = (len(sorted_attrs) + cols - 1) // cols
    fig, ax = plt.subplots(figsize=(14, rows * 1.8 + 2))
    ax.set_xlim(0, cols); ax.set_ylim(-rows - 0.5, 0.5)
    ax.axis('off')

    for i, (tok, score) in enumerate(sorted_attrs):
        col = i % cols
        row = -(i // cols)
        color = cmap(norm(score))
        rect = FancyBboxPatch((col + 0.05, row - 0.4), 0.88, 0.75,
                              boxstyle="round,pad=0.02", facecolor=color, edgecolor='gray', lw=0.5)
        ax.add_patch(rect)
        text_color = 'white' if abs(score) > 0.6 * max_val else 'black'
        ax.text(col + 0.5, row - 0.05, tok,    ha='center', va='center', fontsize=11, fontweight='bold', color=text_color)
        ax.text(col + 0.5, row - 0.28, f"{score:.3f}", ha='center', va='center', fontsize=9, color=text_color)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation='horizontal', fraction=0.05, pad=0.02, aspect=40)
    cbar.set_label("Attribution score (w.r.t. SAFE class)", fontsize=10)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Salvat: {save_path}")


def plot_step_attributions(agg_scores, gt_labels, cat, gt, pred, save_path):
    keys   = list(agg_scores.keys())
    values = [agg_scores[k] for k in keys]
    colors_bars = ['green' if v >= 0 else 'red' for v in values]

    fig, ax = plt.subplots(figsize=(10, max(4, len(keys) * 0.6 + 1)))
    ax.barh(range(len(keys)), values, color=colors_bars, edgecolor='white', height=0.6)
    ax.set_yticks(range(len(keys))); ax.set_yticklabels(keys, fontsize=10)
    ax.invert_yaxis()
    ax.axvline(0, color='gray', linestyle='--', linewidth=1)
    ax.set_xlabel("Aggregated attribution score (positive → SAFE, negative → UNSAFE)", fontsize=10)
    ax.set_title(f"Q3 — Step-level Attribution | {cat} (GT={gt}, Pred={pred})",
                 fontsize=12, color={'TP':'green','TN':'blue','FP':'orange','FN':'purple'}.get(cat,'black'))

    if gt_labels and len(gt_labels) == len(keys) - 1:
        for i, (key, val) in enumerate(zip(keys, values)):
            if key == "Query":
                continue
            step_idx = i - 1
            if step_idx < len(gt_labels):
                gt_step = gt_labels[step_idx]
                label_str = "safe" if gt_step == 0 else "unsafe"
                correct   = (val >= 0) == (gt_step == 0)
                marker    = "✓" if correct else "✗"
                color     = "green" if correct else "red"
                ax.text(val + (0.01 if val >= 0 else -0.01), i,
                        f"GT={label_str} {marker}", va='center',
                        ha='left' if val >= 0 else 'right', fontsize=8, color=color)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Salvat: {save_path}")


## Găsire exemple TP / TN / FP / FN și vizualizare

In [ ]:
def has_gt_variation(row):
    raw = row.get("detailed_label", None)
    if isinstance(raw, str):
        try: raw = json.loads(raw)
        except: return False
    if not isinstance(raw, list) or len(raw) < 2:
        return False
    return 0 in set(raw) and 1 in set(raw)

found      = {"TP": None, "TN": None, "FP": None, "FN": None}
found_rows = {}

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    if all(v is not None for v in found.values()):
        break
    if not has_gt_variation(row):
        continue

    query = str(row.get("query", ""))
    trace = str(row.get("reasoning_trace", ""))
    text  = f"Query: {query}\nReasoning Trace:\n{trace}"
    true  = str(row.get("label", "safe")).strip().lower()

    inp = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        pred_idx = torch.argmax(model(inp["input_ids"], inp["attention_mask"]), -1).item()
    pred = IDX_TO_LABEL[pred_idx]

    is_safe   = (true == "safe")
    pred_safe = (pred == "safe")

    if   is_safe  and pred_safe  and found["TP"] is None: found["TP"] = text; found_rows["TP"] = row
    elif not is_safe and not pred_safe and found["TN"] is None: found["TN"] = text; found_rows["TN"] = row
    elif not is_safe and pred_safe     and found["FP"] is None: found["FP"] = text; found_rows["FP"] = row
    elif is_safe  and not pred_safe    and found["FN"] is None: found["FN"] = text; found_rows["FN"] = row

print({k: "found" if v else "not found" for k, v in found.items()})


In [ ]:
for cat in ["TP", "TN", "FP", "FN"]:
    text = found.get(cat)
    row  = found_rows.get(cat)
    if text is None:
        print(f"{cat}: no example found"); continue

    gt_label  = str(row.get("label", "?")).strip().lower()
    inp_check = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    with torch.no_grad():
        pred_label = IDX_TO_LABEL[torch.argmax(model(inp_check["input_ids"], inp_check["attention_mask"]), -1).item()]

    print(f"\n{cat} (GT={gt_label}, Pred={pred_label})")

    raw_dl = row.get("detailed_label", None)
    if isinstance(raw_dl, str):
        try: raw_dl = json.loads(raw_dl)
        except: raw_dl = None
    if not isinstance(raw_dl, list): raw_dl = None

    word_attrs, raw_attrs, delta = explain_ig(model, tokenizer, text, target_class=0)
    print(f"  Convergence delta: {delta:.4f}")

    agg_scores = aggregate_by_steps(raw_attrs, expected_steps=len(raw_dl) if raw_dl else None)

    heatmap_path = os.path.join(LOG_DIR, f"q3_heatmap_{cat}.png")
    steps_path   = os.path.join(LOG_DIR, f"q3_steps_{cat}.png")
    plot_token_heatmap(word_attrs, cat, gt_label, pred_label, heatmap_path)
    plot_step_attributions(agg_scores, raw_dl, cat, gt_label, pred_label, steps_path)

    if len(agg_scores) > 1 and raw_dl and len(raw_dl) == len(agg_scores) - 1:
        step_scores = [agg_scores[f"Step {i+1}"] for i in range(len(raw_dl))]
        rho, pval   = spearmanr(step_scores, raw_dl)
        print(f"  Spearman rho (IG vs GT step labels): {rho:.3f} (p={pval:.4f})")


## Q4 — Performanță pe buckets de lungime a traiectoriei

In [ ]:
class SafetyDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=MAX_LENGTH):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        row   = self.data.iloc[idx]
        text  = f"Query: {row.get('query','')}\nReasoning Trace:\n{row.get('reasoning_trace','')}"
        enc   = self.tokenizer(text, truncation=True, max_length=self.max_length,
                               padding="max_length", return_tensors="pt")
        label = LABEL_MAP.get(str(row.get('label','safe')).strip().lower(), 0)
        return {'input_ids': enc['input_ids'].flatten(),
                'attention_mask': enc['attention_mask'].flatten(),
                'labels': torch.tensor(label, dtype=torch.long)}


def count_steps(trace):
    return len(re.findall(r'step\s+\d+', str(trace), re.IGNORECASE))

def bucket(n):
    if n <= 5:  return "Short (1-5)"
    if n <= 10: return "Medium (6-10)"
    if n <= 15: return "Long (11-15)"
    return "Very Long (15+)"

test_df2 = test_df.copy()
test_df2['n_steps'] = test_df2['reasoning_trace'].apply(count_steps)
test_df2['bucket']  = test_df2['n_steps'].apply(bucket)

print("Distributie buckets:")
print(test_df2['bucket'].value_counts())


In [ ]:
bucket_order = ["Short (1-5)", "Medium (6-10)", "Long (11-15)", "Very Long (15+)"]

for b in bucket_order:
    sub = test_df2[test_df2['bucket'] == b]
    if len(sub) == 0:
        print(f"{b}: no examples"); continue

    ds     = SafetyDataset(sub, tokenizer)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False)
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            preds.extend(torch.argmax(logits, -1).cpu().numpy())
            labels.extend(batch['labels'].numpy())

    acc = accuracy_score(labels, preds)
    mf1 = f1_score(labels, preds, average='macro', zero_division=0)
    print(f"{b:<22} | n={len(sub):>4} | Acc={acc*100:.2f}% | Macro-F1={mf1:.4f}")
